In [4]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import torch
tqdm.pandas()

class Solver:
    
    def __init__(self,model="facebook/bart-large-mnli"):
        self.model = model
        
    def solve(self, text, labels):
        classifier = pipeline(
            "zero-shot-classification",
            model = self.model
        )
        result = classifier(text, labels)
        return result

d:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\venv_mcq\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
device_id = 0 if torch.cuda.is_available() else -1

In [6]:
device_id

-1

In [7]:
test_df = pd.read_csv(r'D:\Placement Prep\Projects\MCQ Solver\MCQ-Solver\dataset\test.csv')

In [11]:
def get_top3(prompt, options, solver):

    result = solver.solve(prompt, options)
    winning_option_texts = result['labels']
    keys = ['A', 'B', 'C', 'D', 'E']
    top3_keys = []
    for text in winning_option_texts[:3]:
        original_index = options.index(text)
        top3_keys.append(keys[original_index])
        
    return " ".join(top3_keys)

In [14]:
test_df = test_df.iloc[0:3]

In [15]:
solver = Solver()
import mapply

# 1. Initialize mapply (adds .mapply to pandas)
mapply.init(
    n_workers=-1,      # Use all available CPU cores
    progressbar=True   # Show the progress bar
)
test_df['Prediction'] = test_df.progress_apply(lambda x: get_top3(x['prompt'],[x['A'],x['B'],x['C'],x['D'],x['E']],solver), axis = 1)

100%|██████████| 3/3 [00:15<00:00,  5.09s/it]


In [16]:
test_df

,id,prompt,A,B,C,D,E,Prediction
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...",D C B
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi...",E B D
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...,D A B


In [ ]:
sub_df = test_df[['id','Prediction']]
sub_df.columns = ['ID','Prediction']
sub_df.to_csv('submission.csv',index=False)